# Weather Analysis with PySpark in Jupyter

This notebook demonstrates running the weather_analysis package with PySpark directly in a Jupyter cell.

In [3]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://archive.apache.org/dist/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
%pip install -q findspark
%pip install pyspark
%pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Our First Spark Example") \
       .getOrCreate()

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,955 kB]  
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,842 kB]3m
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,954 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-upda

In [4]:
# Install Java, Spark and Python deps
!apt-get update -qq
!apt-get install -y openjdk-8-jdk-headless -qq
!wget -q https://archive.apache.org/dist/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar -xzf spark-3.2.1-bin-hadoop3.2.tgz
!pip install pyspark matplotlib findspark

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
os.environ['SPARK_HOME'] = '/content/spark-3.2.1-bin-hadoop3.2'
import findspark
findspark.init(os.environ['SPARK_HOME'])

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [11]:
# Add project to Python path to import the weather_analysis package
import sys
sys.path.insert(0, '/home/david/CS3270/CS3270')

# Import weather analysis functions
from weather_analysis import (
    analyze_rain_patterns_df,
    max_temperature_df,
    min_temperature_df,
    total_rainfall_df,
    count_rainy_days_df,
    temperature_range_stats_df,
)
from weather_analysis.logger_config import setup_logger

logger = setup_logger('notebook')

In [13]:
# Load weather data CSV as a Spark DataFrame
file_path = "Weather Training Data.csv"

df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)

# Cast RainToday and RainTomorrow to strings to handle mixed types
from pyspark.sql.functions import col
df = df.withColumn("RainToday", col("RainToday").cast("string")) \
       .withColumn("RainTomorrow", col("RainTomorrow").cast("string"))

print(f"Dataset loaded: {df.count()} rows, {len(df.columns)} columns")
df.printSchema()

Dataset loaded: 99516 rows, 23 columns
root
 |-- row ID: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- MinTemp: double (nullable = true)
 |-- MaxTemp: double (nullable = true)
 |-- Rainfall: double (nullable = true)
 |-- Evaporation: double (nullable = true)
 |-- Sunshine: double (nullable = true)
 |-- WindGustDir: string (nullable = true)
 |-- WindGustSpeed: integer (nullable = true)
 |-- WindDir9am: string (nullable = true)
 |-- WindDir3pm: string (nullable = true)
 |-- WindSpeed9am: integer (nullable = true)
 |-- WindSpeed3pm: integer (nullable = true)
 |-- Humidity9am: integer (nullable = true)
 |-- Humidity3pm: integer (nullable = true)
 |-- Pressure9am: double (nullable = true)
 |-- Pressure3pm: double (nullable = true)
 |-- Cloud9am: integer (nullable = true)
 |-- Cloud3pm: integer (nullable = true)
 |-- Temp9am: double (nullable = true)
 |-- Temp3pm: double (nullable = true)
 |-- RainToday: string (nullable = true)
 |-- RainTomorrow: string (nullable = t

In [14]:
# Distributed analysis on Spark DataFrame
print("=" * 60)
print("WEATHER ANALYSIS - SPARK DATAFRAME OPERATIONS")
print("=" * 60)

print("\n-- Rain Pattern Analysis --")
rain_patterns = analyze_rain_patterns_df(df)
print(rain_patterns)

print("\n-- Temperature Extremes --")
max_temp = max_temperature_df(df)
min_temp = min_temperature_df(df)
print(f"Highest recorded temp: {max_temp}°C")
print(f"Lowest recorded temp:  {min_temp}°C")

print("\n-- Total Rainfall --")
total_rainfall = total_rainfall_df(df)
print(f"Total rainfall: {total_rainfall:.1f} mm")

print("\n-- Rainy Days Count --")
rainy_days = count_rainy_days_df(df)
print(f"Days with rain: {rainy_days}")

print("\n-- Temperature Range Statistics --")
tr_stats = temperature_range_stats_df(df)
if tr_stats['count']:
    print(f"Average daily range: {tr_stats['avg']:.2f}°C (n={tr_stats['count']})")
    print(f"Max daily range: {tr_stats['max']:.2f}°C")
    print(f"Min daily range: {tr_stats['min']:.2f}°C")

WEATHER ANALYSIS - SPARK DATAFRAME OPERATIONS

-- Rain Pattern Analysis --
{'rain_today': 22056, 'rain_tomorrow': 0, 'consecutive_rain': 0, 'total_days': 99516}

-- Temperature Extremes --
Highest recorded temp: 48.1°C
Lowest recorded temp:  -8.5°C

-- Total Rainfall --
Total rainfall: 231859.9 mm

-- Rainy Days Count --
Days with rain: 22056

-- Temperature Range Statistics --
Average daily range: 11.04°C (n=98907)
Max daily range: 31.20°C
Min daily range: 0.00°C
